In [0]:
# No spark.conf.set() needed!
# Unity Catalog + External Location handles auth

base_path = "abfss://sapdata@adlsloki.dfs.core.windows.net"

# Test connection
files = dbutils.fs.ls(f"{base_path}/Bronze")
print("✅ Connected! Bronze files:")
for f in files:
    print(f"  → {f.name}")

In [0]:
# ── Unity Catalog Setup with Storage Location ──

# External Location URL
adls_location = "abfss://sapdata@adlsloki.dfs.core.windows.net/unity-catalog"

# Catalog create with location
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS sap_catalog
    MANAGED LOCATION '{adls_location}'
""")

spark.sql("USE CATALOG sap_catalog")

# Schemas create
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze_schema")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver_schema")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_schema")

print("✅ Unity Catalog ready!")
display(spark.sql("SHOW SCHEMAS IN sap_catalog"))

In [0]:
# ── Cell 3: Paths + Table List ─────────────────
base_path   = "abfss://sapdata@adlsloki.dfs.core.windows.net"
bronze_path = f"{base_path}/Bronze"
table_list  = ["vbak","vbap","kna1","mara","bkpf","bseg"]

print(f"Bronze path: {bronze_path}")
print(f"Tables: {table_list}")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_ingestion")

def ingest_to_bronze(table_name):
    try:
        source = f"{bronze_path}/{table_name}.csv"
        
        # Read CSV
        df = (spark.read
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .csv(source))
        
        count = df.count()
        
        # Add metadata columns
        # ✅ UC-compatible: no input_file_name()
        df_meta = (df
            .withColumn("_ingestion_ts",  current_timestamp())
            .withColumn("_source_file",   lit(source))
            .withColumn("_source_system", lit("SAP"))
            .withColumn("_table_name",    lit(table_name)))
        
        # Write as Delta managed table
        target = f"sap_catalog.bronze_schema.{table_name}_raw"
        
        (df_meta.write
               .format("delta")
               .mode("overwrite")
               .option("overwriteSchema","true")
               .saveAsTable(target))
        
        print(f"✅ {table_name} → {count:,} rows")
        return {"table":table_name,"status":"SUCCESS","rows":count}
        
    except Exception as e:
        print(f"❌ {table_name} failed: {str(e)}")
        return {"table":table_name,"status":"FAILED","error":str(e)}

print("✅ Function ready!")

In [0]:
# ── Cell 5: Execute ────────────────────────────
results = []
for table in table_list:
    result = ingest_to_bronze(table)
    results.append(result)

print("\n" + "="*40)
print("BRONZE INGESTION SUMMARY")
print("="*40)
success = [r for r in results if r["status"]=="SUCCESS"]
failed  = [r for r in results if r["status"]=="FAILED"]
print(f"✅ Success: {len(success)}/6 tables")
print(f"❌ Failed:  {len(failed)}/6 tables")